# 🏰 Citadel Model Aliases — End-to-End Validation

This notebook validates the **shared `resolve-model-alias` policy fragment** that exposes a consistent alias-routing experience across all 3 LLM endpoints in the Citadel Governance Hub:

| API | Where the model goes |
|-----|----------------------|
| **Universal LLM API** | `model` field in JSON request body |
| **Azure OpenAI API** | `/openai/deployments/{model}/...` URL path |
| **Unified AI API** | wildcard route — body or path depending on api-type |

### What this notebook does

1. **Initialize** from either `azd` env vars (one-shot) or manual variables
2. **Onboard backends** with **two model aliases** — one `priority`-routed, one `weighted`-routed
3. **Create an access contract** scoped to the alias names (least-privilege RBAC)
4. **Test direct model selection** to confirm aliases don't break the standard path
5. **Test alias resolution** across all 3 APIs and inspect the `UAIG-*` debug headers to see exactly what the gateway did
6. **Test weighted distribution** by issuing N requests through a `weighted` alias and tallying the resolved models

> **Pre-requisite**: A deployed Citadel Governance Hub. If you used `azd` to deploy, the next cell can pull `LLM_BACKEND_CONFIG` straight from your active `azd` environment.

<a id='0'></a>
### 0️⃣ Initialize Notebook Variables

**Choose ONE initialization mode** by setting `init_from_azd`:

- `True` — autoload `governance_hub_resource_group`, `location`, and `llm_backends_config` from your active `azd` environment via `utils.load_azd_env(...)` (which wraps `azd env get-value ...`). This works when the accelerator was deployed with `azd up` — see the [Initializing Variables from azd Environment](README.md#initializing-variables-from-azd-environment) section in the README for the full per-notebook variable map.
- `False` — fill the `REPLACE` values manually below and skip the `azd` lookup entirely.

The two **alias definitions** under `model_aliases` are the focus of this validation:

- `adv-gpt` (`priority` strategy) — first underlying model wins; remaining models act as cross-model fallback.
- `gpt-blend` (`weighted` strategy) — random-weighted picking. Useful for A/B-testing model swaps.

In [ ]:
import os
import sys, json, requests, time
from collections import Counter
sys.path.insert(1, '../shared')
import utils
from apimtools import APIMClientTool

# ============================================================================
# 🔧 INITIALIZATION MODE
# ----------------------------------------------------------------------------
# `init_from_azd = True` pulls deployment-time outputs from your active azd
# environment (set by `azd up`). Manual values below act as defaults /
# fallbacks when an azd value is missing. Set False to fill the REPLACE values
# manually and skip the azd lookup entirely.
# ============================================================================
init_from_azd = True

# ============================================================================
# 🔧 MANUAL CONFIG (used when init_from_azd = False, or as fallback)
# ============================================================================
governance_hub_resource_group = "REPLACE"   # e.g. "rg-ai-hub-citadel-dev"
location                      = "REPLACE"   # e.g. "swedencentral"
llm_backends_config           = []          # paste the JSON array if not using azd

# ============================================================================
# 🔁 azd ENVIRONMENT OVERRIDES
# ----------------------------------------------------------------------------
# Auto-populated from the active azd environment when `init_from_azd = True`:
#   AZURE_RESOURCE_GROUP / GOVERNANCE_HUB_RESOURCE_GROUP → governance_hub_resource_group
#   AZURE_LOCATION       / LOCATION                      → location
#   LLM_BACKEND_CONFIG   / LLM_BACKENDS_CONFIG (JSON)    → llm_backends_config
# ============================================================================
if init_from_azd:
    utils.print_info("Loading configuration from azd environment...")
    loaded = utils.load_azd_env({
        "resource_group": ["AZURE_RESOURCE_GROUP", "GOVERNANCE_HUB_RESOURCE_GROUP"],
        "location":       ["AZURE_LOCATION", "LOCATION"],
        "llm_backends":   (["LLM_BACKEND_CONFIG", "LLM_BACKENDS_CONFIG"], "json"),
    }, verbose=False)

    if "resource_group" in loaded:  governance_hub_resource_group = loaded["resource_group"]
    if "location"       in loaded:  location                      = loaded["location"]
    if "llm_backends"   in loaded:  llm_backends_config           = loaded["llm_backends"]

    utils.print_ok(f"Resource group : {governance_hub_resource_group}")
    utils.print_ok(f"Location       : {location}")
    utils.print_ok(f"Backends loaded: {len(llm_backends_config)}")

# ============================================================================
# 🔧 ALIAS DEFINITIONS — the focus of this validation
# ============================================================================
# These two aliases are used by the access contract and the test cells below.
# Adjust the underlying `models` lists to match models actually present in
# your `llm_backends_config` (the next cell prints the list of models found).
model_aliases = [
    {
        "name": "adv-gpt",
        "models": ["gpt-5.2", "gpt-5.4-mini", "gpt-4.1"],
        "strategy": "priority",
    },
    {
        "name": "gpt-blend",
        "models": ["gpt-5.4-mini", "gpt-4.1"],
        "strategy": "weighted",
        "weights": [70, 30],
    },
]

# Direct model used in the "control" test (must exist in llm_backends_config)
direct_test_model = "gpt-4.1"

# ============================================================================
# 🔧 API VERSION
# ============================================================================
inference_api_version = "2024-05-01-preview"

# ============================================================================
# 🔧 OPTIONAL — Only needed for backends that use Key Vault / AWS / Foundry
# ============================================================================
key_vault_name = ""
aws_access_key = ""
aws_secret_key = ""
aws_region     = ""

# Validate prerequisites
if governance_hub_resource_group == "REPLACE" or location == "REPLACE":
    utils.print_warning("⚠️  governance_hub_resource_group / location not set — fix before continuing.")
if not llm_backends_config:
    utils.print_warning("⚠️  llm_backends_config is empty — provide via azd or paste it manually.")

# Quick visibility into what models will be available
all_models = sorted({
    m["name"] if isinstance(m, dict) else m
    for b in llm_backends_config for m in b.get("supportedModels", [])
})
utils.print_info(f"Models discovered in backends_config: {', '.join(all_models) if all_models else '(none)'}")
for alias in model_aliases:
    missing = [m for m in alias["models"] if m not in all_models]
    if missing:
        utils.print_warning(f"⚠️  Alias '{alias['name']}' references missing model(s): {missing}")
if direct_test_model not in all_models:
    utils.print_warning(f"⚠️  direct_test_model '{direct_test_model}' is not in backends_config.")

utils.print_ok("Initialization completed.")

<a id='1'></a>
### 1️⃣ Verify Azure CLI and Initialize APIM Client

In [ ]:
output = utils.run("az account show", "Retrieved az account", "Failed to get the current az account")
if output.success and output.json_data:
    current_user    = output.json_data['user']['name']
    tenant_id       = output.json_data['tenantId']
    subscription_id = output.json_data['id']
    utils.print_info(f"User           : {current_user}")
    utils.print_info(f"Tenant         : {tenant_id}")
    utils.print_info(f"Subscription   : {subscription_id}")

apimClientTool = APIMClientTool(governance_hub_resource_group)
apimClientTool.initialize()
apim_resource_name        = apimClientTool.apim_resource_name
apim_resource_gateway_url = str(apimClientTool.apim_resource_gateway_url)
utils.print_ok(f"APIM Resource: {apim_resource_name}")
utils.print_ok(f"Gateway URL  : {apim_resource_gateway_url}")

# Discover the user-assigned managed identity
managed_identity_info               = apimClientTool.get_managed_identity_info()
managed_identity_client_id          = managed_identity_info.get('clientId')
managed_identity_name               = managed_identity_info.get('name')
managed_identity_resource_group     = managed_identity_info.get('resourceGroup') or governance_hub_resource_group
utils.print_info(f"Managed Identity: {managed_identity_name} (clientId={managed_identity_client_id})")

<a id='2'></a>
### 2️⃣ Generate the Onboarding `.bicepparam` (with both aliases)

This cell writes a self-contained `.bicepparam` to `bicep/infra/llm-backend-onboarding/llm-backends-aliases-validation.bicepparam` containing:

- The full `llmBackendConfig` array loaded above
- The two aliases (`adv-gpt` priority + `gpt-blend` weighted)
- Optional `keyVaultName` / AWS named values when supplied

The next cell deploys it. The deployment regenerates the **shared `resolve-model-alias` policy fragment** with the two aliases inlined as a static `JObject` (used by Azure OpenAI / Universal LLM APIs) AND publishes them in `metadata-config` (used by Unified AI via the cache manager).

In [ ]:
bicep_dir   = "../bicep/infra/llm-backend-onboarding"
params_file = os.path.join(bicep_dir, "llm-backends-aliases-validation.bicepparam")

def fmt_model(m):
    if isinstance(m, str):
        return f"{{ name: '{m}' }}"
    parts = [f"name: '{m['name']}'"]
    for k in ('sku', 'modelFormat', 'modelVersion', 'retirementDate', 'apiVersion', 'inferenceApiVersion'):
        if k in m: parts.append(f"{k}: '{m[k]}'")
    for k in ('capacity', 'timeout'):
        if k in m: parts.append(f"{k}: {m[k]}")
    return "{ " + ", ".join(parts) + " }"

def fmt_backend(b):
    models = "\n      ".join(fmt_model(m) for m in b['supportedModels'])
    auth   = (f"    authType: '{b['authType']}'\n"   if 'authType'   in b else
              f"    authScheme: '{b['authScheme']}'\n" if 'authScheme' in b else "")
    return (f"  {{\n"
            f"    backendId: '{b['backendId']}'\n"
            f"    backendType: '{b['backendType']}'\n"
            f"    endpoint: '{b['endpoint']}'\n"
            f"{auth}"
            f"    supportedModels: [\n      {models}\n    ]\n"
            f"    priority: {b.get('priority', 1)}\n"
            f"    weight: {b.get('weight', 100)}\n"
            f"  }}")

def fmt_alias(a):
    parts = [f"name: '{a['name']}'"]
    parts.append("models: [ " + ", ".join(f"'{m}'" for m in a['models']) + " ]")
    parts.append(f"strategy: '{a.get('strategy', 'priority')}'")
    if 'weights' in a:
        parts.append("weights: [ " + ", ".join(str(w) for w in a['weights']) + " ]")
    return "  { " + ", ".join(parts) + " }"

backends_str = "\n".join(fmt_backend(b) for b in llm_backends_config)
aliases_str  = "\n".join(fmt_alias(a)   for a in model_aliases)

kv_block  = f"\nparam keyVaultName = '{key_vault_name}'\n" if key_vault_name else ""
aws_block = (f"\nparam awsAccessKey = '{aws_access_key}'\nparam awsSecretKey = '{aws_secret_key}'\nparam awsRegion = '{aws_region}'\n"
             if (aws_access_key or aws_secret_key or aws_region) else "")

params_content = f"""using './main.bicep'

// Generated by citadel-model-aliases-tests.ipynb @ {time.strftime('%Y-%m-%d %H:%M:%S')}

param apim = {{
  subscriptionId: '{subscription_id}'
  resourceGroupName: '{governance_hub_resource_group}'
  name: '{apim_resource_name}'
}}

param apimManagedIdentity = {{
  subscriptionId: '{subscription_id}'
  resourceGroupName: '{managed_identity_resource_group}'
  name: '{managed_identity_name}'
}}

param llmBackendConfig = [
{backends_str}
]

param configureCircuitBreaker = true

// Two aliases — the focus of this validation
param modelAliases = [
{aliases_str}
]
{kv_block}{aws_block}"""

with open(params_file, 'w') as f:
    f.write(params_content)

utils.print_ok(f"Wrote {params_file}")
print("\n" + "-"*60)
print(params_content)
print("-"*60)

<a id='3'></a>
### 3️⃣ Deploy the Onboarding (creates / refreshes the `resolve-model-alias` fragment)

In [ ]:
deployment_name = f"alias-validation-{time.strftime('%Y%m%d%H%M%S')}"
template_file   = os.path.join(bicep_dir, "main.bicep")
deployment_cmd  = (
    f"az deployment sub create --name {deployment_name} "
    f"--location {location} --template-file {template_file} --parameters {params_file}"
)
output = utils.run(
    deployment_cmd,
    f"Deployment '{deployment_name}' succeeded",
    f"Deployment '{deployment_name}' failed",
)
if output.success and output.json_data:
    outputs = output.json_data.get('properties', {}).get('outputs', {})
    if 'policyFragments' in outputs:
        fragments = outputs['policyFragments']['value']
        utils.print_ok("Deployed policy fragments:")
        for k, v in fragments.items():
            print(f"  • {k} → {v}")
        if 'resolveModelAlias' in fragments:
            utils.print_ok("✓ resolve-model-alias fragment was (re)deployed with the two aliases inlined.")

<a id='4'></a>
### 4️⃣ Create an Access Contract Scoped to the Aliases

This deploys a single **access contract** (APIM product + subscription) where `allowedModels` lists **only the alias names** plus the direct-test model. This proves that:

- A client can be authorized purely against an alias name (the underlying real models stay hidden behind the abstraction).
- Direct model selection still works for explicitly-allowed real models.
- The product policy turns on `enableResponseHeaders` so we can inspect the `UAIG-*` debug headers in the test cells.

In [ ]:
timestamp = time.strftime('%Y%m%d%H%M%S')
contract = {
    "name": f"alias-validation-contract-{timestamp}",
    "business_unit": "Validation",
    "use_case_name": "AliasTests",
    "environment": "DEV",
    # RBAC: alias names + the direct-test model only
    "allowed_models": [a["name"] for a in model_aliases] + [direct_test_model],
    "tokens_per_minute": 5000,
    "token_quota": 100000,
    "token_quota_period": "Daily",
    "description": "Access contract for model alias validation",
}
utils.print_info(f"Contract: {contract['name']}")
utils.print_info(f"Allowed models: {contract['allowed_models']}")

# Build per-contract product policy XML
allowed_csv = ",".join(contract['allowed_models'])
policy_xml = f"""<policies>
    <inbound>
        <base />
        <!-- Extract requested model from path or body -->
        <include-fragment fragment-id="set-llm-requested-model" />

        <!-- Model RBAC against alias name (resolve-model-alias runs AFTER this) -->
        <set-variable name="allowedModels" value="{allowed_csv}" />
        <include-fragment fragment-id="validate-model-access" />

        <!-- Capacity allocation -->
        <llm-token-limit counter-key="@(context.Subscription.Id)"
                         tokens-per-minute="{contract['tokens_per_minute']}"
                         estimate-prompt-tokens="false"
                         token-quota="{contract['token_quota']}"
                         token-quota-period="{contract['token_quota_period']}" />

        <!-- Turn on UAIG-* debug headers so we can inspect alias resolution at runtime -->
        <set-variable name="enableResponseHeaders" value="@(true)" />
    </inbound>
    <backend><base /></backend>
    <outbound><base /></outbound>
    <on-error><base /></on-error>
</policies>"""

contract_dir = os.path.join(
    "../bicep/infra/citadel-access-contracts", "contracts",
    f"{contract['business_unit'].lower()}-{contract['use_case_name'].lower()}",
    contract['environment'].lower(),
)
os.makedirs(contract_dir, exist_ok=True)
policy_path = os.path.join(contract_dir, "ai-product-policy.xml")
with open(policy_path, 'w') as f:
    f.write(policy_xml)

param_path = os.path.join(contract_dir, "main.bicepparam")
param_content = f"""using '../../../main.bicep'

param apim = {{
  subscriptionId: '{subscription_id}'
  resourceGroupName: '{governance_hub_resource_group}'
  name: '{apim_resource_name}'
}}

// Key Vault is not used by this contract
param keyVault = {{
  subscriptionId: '00000000-0000-0000-0000-000000000000'
  resourceGroupName: 'placeholder'
  name: 'placeholder'
}}
param useTargetAzureKeyVault = false

param useCase = {{
  businessUnit: '{contract['business_unit']}'
  useCaseName: '{contract['use_case_name']}'
  environment: '{contract['environment']}'
}}

param apiNameMapping = {{
  LLM: ['universal-llm-api', 'azure-openai-api', 'unified-ai-api']
}}

param services = [
  {{
    code: 'LLM'
    endpointSecretName: 'ALIAS-VAL-ENDPOINT'
    apiKeySecretName: 'ALIAS-VAL-KEY'
    policyXml: loadTextContent('ai-product-policy.xml')
  }}
]

param productTerms = '{contract['description']}'
param useTargetFoundry = false
"""
with open(param_path, 'w') as f:
    f.write(param_content)

utils.print_ok(f"Wrote contract template at {param_path}")

# Deploy the contract
contract_template = "../bicep/infra/citadel-access-contracts/main.bicep"
out = utils.run(
    f"az deployment sub create --name {contract['name']} --location {location} "
    f"--template-file {contract_template} --parameters {param_path}",
    f"Contract '{contract['name']}' deployed",
    f"Contract '{contract['name']}' deployment failed",
)
if out.success:
    apimClientTool.initialize()  # refresh subscriptions
    utils.print_ok(f"APIM now has {len(apimClientTool.apim_subscriptions)} subscription(s).")

<a id='5'></a>
### 5️⃣ Resolve the Contract's API Key

We pick the subscription belonging to the access contract just deployed and use its primary key for the upcoming tests.

In [ ]:
# Find the subscription created for this contract. Naming convention: LLM-<BU>-<UC>-<ENV>...
expected_prefix = f"LLM-{contract['business_unit']}-{contract['use_case_name']}-{contract['environment']}"
matching = [s for s in apimClientTool.apim_subscriptions if str(s.get("name", "")).startswith(expected_prefix)]
if matching:
    api_key  = matching[0]["key"]
    api_sub  = matching[0]["name"]
    utils.print_ok(f"Using subscription '{api_sub}'")
elif apimClientTool.apim_subscriptions:
    api_key = apimClientTool.apim_subscriptions[-1]["key"]
    api_sub = apimClientTool.apim_subscriptions[-1]["name"]
    utils.print_warning(f"Could not find subscription with prefix {expected_prefix} — falling back to '{api_sub}'")
else:
    api_key = None
    api_sub = None
    utils.print_error("No APIM subscriptions found. Did the contract deploy succeed?")

<a id='6'></a>
### 6️⃣ Discover API Endpoints (Universal LLM, Azure OpenAI, Unified AI)

In [ ]:
endpoints = {}

for label, path_filter in [
    ("universal", "models"),
    ("openai",    "openai"),
    ("unified",   "unified-ai"),
]:
    try:
        apimClientTool.discover_api(path_filter)
        endpoints[label] = str(apimClientTool.azure_endpoint)
        utils.print_ok(f"{label:10s} → {endpoints[label]}")
    except Exception as e:
        endpoints[label] = None
        utils.print_warning(f"{label:10s} not deployed: {e}")

<a id='helpers'></a>
### 🧰 Test Helpers — Inspect `UAIG-*` Response Headers

All tests below print the request response code, the resolved model from the response body, and the relevant `UAIG-*` headers so you can see exactly what the gateway did.

Headers we care about:

| Header | Meaning |
|--------|---------|
| `UAIG-Model-Id` | Model that was actually routed to (post alias resolution) |
| `UAIG-Alias` | Original alias name the client sent (only present when alias was used) |
| `UAIG-Resolved-Model` | Real model the alias resolved to |
| `UAIG-Backend` | Backend pool / backend that served the request |
| `UAIG-API-Type` | Detected API type (Unified AI only) |
| `UAIG-Final-Path` | Reconstructed backend path (Unified AI only) |
| `UAIG-Auth-Type` | Auth method enforced by `security-handler` (Unified AI only) |
| `UAIG-Cache-Operation` | `cache-hit` / `cache-miss` for `metadata-config` |
| `UAIG-Request-Id` | APIM request id for log correlation |

> Headers prefixed with `UAIG-` only appear when the product policy sets `enableResponseHeaders=true` (our contract does).

In [ ]:
UAIG_HEADERS = [
    "UAIG-Model-Id", "UAIG-Alias", "UAIG-Resolved-Model", "UAIG-Backend",
    "UAIG-API-Type", "UAIG-Final-Path", "UAIG-Auth-Type",
    "UAIG-Cache-Operation", "UAIG-Request-Id",
]

TEST_MESSAGES = [
    {"role": "system", "content": "You are a helpful assistant. Be concise."},
    {"role": "user",   "content": "Reply with the single word OK."},
]

def _print_uaig(headers):
    found = [(h, headers.get(h)) for h in UAIG_HEADERS if headers.get(h) is not None]
    if not found:
        print("   (no UAIG-* headers in response — is enableResponseHeaders set?)")
        return
    width = max(len(h) for h, _ in found)
    for h, v in found:
        print(f"   {h.ljust(width)} = {v}")

def call_universal(model_or_alias):
    if not endpoints.get("universal") or not api_key: return None
    url = f"{endpoints['universal']}models/chat/completions?api-version={inference_api_version}"
    return requests.post(url, headers={"api-key": api_key},
                         json={"model": model_or_alias, "messages": TEST_MESSAGES}, timeout=60)

def call_openai(model_or_alias):
    if not endpoints.get("openai") or not api_key: return None
    url = f"{endpoints['openai']}openai/deployments/{model_or_alias}/chat/completions?api-version={inference_api_version}"
    return requests.post(url, headers={"api-key": api_key},
                         json={"messages": TEST_MESSAGES}, timeout=60)

def call_unified(model_or_alias):
    if not endpoints.get("unified") or not api_key: return None
    url = f"{endpoints['unified']}models/chat/completions?api-version={inference_api_version}"
    return requests.post(url, headers={"api-key": api_key},
                         json={"model": model_or_alias, "messages": TEST_MESSAGES}, timeout=60)

def report(label, response, expected_alias=None):
    print(f"\n  📡 {label}")
    if response is None:
        utils.print_warning("     skipped (endpoint or api key not available)")
        return
    print(f"     status: {response.status_code}")
    if response.status_code == 200:
        try:
            data = response.json()
            resolved = data.get("model", "?")
            reply    = data.get("choices", [{}])[0].get("message", {}).get("content", "").strip()
            utils.print_ok(f"     resolved model in body: '{resolved}'  reply: '{reply}'")
        except Exception:
            pass
    else:
        print(f"     body: {response.text[:200]}")
    _print_uaig(response.headers)
    if expected_alias:
        actual_alias = response.headers.get("UAIG-Alias")
        if actual_alias and actual_alias.lower() == expected_alias.lower():
            utils.print_ok(f"     ✓ UAIG-Alias matches expected alias '{expected_alias}'")
        elif response.status_code == 200 and not actual_alias:
            utils.print_warning(f"     ⚠ UAIG-Alias header missing (response headers may not be enabled)")

utils.print_ok("Helpers loaded.")

<a id='discovery'></a>
### 🔎 Discovery Test — `GET /deployments` Honors `allowedModels`

Calls `GET /deployments` on each API. The `get-available-models` policy fragment filters by the contract's `allowedModels`, so the response should contain **only the alias names + the direct-test model** — NOT every backend-deployed model.

Specifically we assert:

- ✅ The two alias names (`adv-gpt`, `gpt-blend`) appear as `type: "alias"` entries with `properties.capabilities.description` describing their underlying real models and routing strategy.
- ✅ `direct_test_model` appears as a real model entry (`type: "ai-foundry"` / `azure-openai` / etc.).
- ✅ Underlying real models that are NOT explicitly in `allowedModels` (e.g. `gpt-5.2`, `gpt-5.4-mini`) are **filtered out** even though they back the aliases — proving RBAC at discovery is consistent with RBAC at chat-completion time.


In [ ]:
# Discovery test: GET /deployments on each API. Asserts the response is filtered
# to the contract's `allowedModels` and that aliases appear as first-class entries.

# /deployments URL by API:
#   Universal LLM API → {gateway}/models/deployments
#   Azure OpenAI API  → {gateway}/openai/deployments
#   Unified AI API    → {gateway}/unified-ai/deployments  (named operation at API root)
DEPLOYMENT_PATHS = {
    "universal": "models/deployments",
    "openai":    "openai/deployments",
    "unified":   "unified-ai/deployments",
}

def call_deployments(api_label):
    """Invoke GET /deployments on the given API and return (url, response)."""
    base = endpoints.get(api_label)
    if not base or not api_key:
        return None, None
    url = f"{base}{DEPLOYMENT_PATHS[api_label]}?api-version={inference_api_version}"
    return url, requests.get(url, headers={"api-key": api_key}, timeout=30)

expected_allowed = set(contract["allowed_models"])
alias_names      = {a["name"] for a in model_aliases}
print(f"Expected entries (from allowedModels): {sorted(expected_allowed)}")
print(f"  of which are aliases               : {sorted(alias_names & expected_allowed)}")
print(f"  models that should be HIDDEN       : {sorted(set(all_models) - expected_allowed)}\n")

for api_label, label in [("universal", "Universal LLM API"),
                          ("openai",    "Azure OpenAI API"),
                          ("unified",   "Unified AI API")]:
    print(f"📡 {label}")
    url, response = call_deployments(api_label)
    if response is None:
        utils.print_warning(f"   skipped ({api_label} endpoint not available)")
        continue
    print(f"   GET {url}")
    print(f"   status: {response.status_code}")
    if response.status_code != 200:
        utils.print_error(f"   body: {response.text[:300]}")
        continue

    try:
        data = response.json()
    except Exception as e:
        utils.print_error(f"   could not parse JSON: {e}")
        continue

    deployments    = data.get("value", [])
    returned_names = {d.get("name") for d in deployments}
    print(f"   returned {len(deployments)} entries: {sorted(returned_names)}")

    # Assertions
    missing = expected_allowed - returned_names
    leaked  = returned_names - expected_allowed
    if missing:
        utils.print_error(f"   ✗ missing expected entries: {sorted(missing)}")
    else:
        utils.print_ok("   ✓ all expected entries present")
    if leaked:
        utils.print_error(f"   ✗ entries leaked past allowedModels filter: {sorted(leaked)}")
    else:
        utils.print_ok("   ✓ no leaked entries — RBAC honored")

    # Inspect alias entries: type, format, and the description on capabilities
    for d in deployments:
        if d.get("name") in alias_names:
            d_type      = d.get("type", "?")
            model_block = d.get("properties", {}).get("model", {})
            d_format    = model_block.get("format", "?")
            description = d.get("properties", {}).get("capabilities", {}).get("description")
            print(f"\n   🔁 alias entry: {d.get('name')}")
            print(f"      type       : {d_type}")
            print(f"      format     : {d_format}")
            print(f"      description: {description}")
            if d_type == "alias" and d_format == "Alias" and description:
                utils.print_ok("      ✓ alias entry shape correct")
            else:
                utils.print_warning("      ⚠ alias entry missing expected type/format/description fields")
    print()


<a id='7'></a>
### 7️⃣ Control Test — Direct Model Selection (no alias)

Confirms that asking for a real model name still routes correctly across all 3 APIs and that the `resolve-model-alias` fragment is a no-op (no `UAIG-Alias` header should appear).

In [ ]:
print(f"Direct model: {direct_test_model}")
report("Universal LLM API", call_universal(direct_test_model))
report("Azure OpenAI API",  call_openai(direct_test_model))
report("Unified AI API",    call_unified(direct_test_model))

<a id='8'></a>
### 8️⃣ Alias Test — `priority` Strategy

Calls `adv-gpt` (alias) on each API. Expected behavior:

- Response body's `model` field reflects the **first underlying model** in the alias list (e.g. `gpt-5.2`).
- `UAIG-Alias` = `adv-gpt`, `UAIG-Resolved-Model` = the resolved real model.
- All three APIs resolve to the **same** real model on the same call (deterministic priority).

In [ ]:
alias_priority = next(a for a in model_aliases if a["strategy"] == "priority")
alias_name = alias_priority["name"]
print(f"Alias: {alias_name} (priority over {alias_priority['models']})")
report("Universal LLM API", call_universal(alias_name), expected_alias=alias_name)
report("Azure OpenAI API",  call_openai(alias_name),    expected_alias=alias_name)
report("Unified AI API",    call_unified(alias_name),   expected_alias=alias_name)

<a id='9'></a>
### 9️⃣ Alias Test — `weighted` Strategy (single call)

One call against `gpt-blend`. The resolved model varies between calls because selection is random-weighted.

In [ ]:
alias_weighted = next((a for a in model_aliases if a["strategy"] == "weighted"), None)
if not alias_weighted:
    utils.print_warning("No weighted alias defined — skipping.")
else:
    alias_name = alias_weighted["name"]
    print(f"Alias: {alias_name} (weighted: models={alias_weighted['models']} weights={alias_weighted.get('weights')})")
    report("Universal LLM API", call_universal(alias_name), expected_alias=alias_name)
    report("Azure OpenAI API",  call_openai(alias_name),    expected_alias=alias_name)
    report("Unified AI API",    call_unified(alias_name),   expected_alias=alias_name)

<a id='10'></a>
### 🔟 Distribution Test — `weighted` Strategy over N Requests

Sends N requests through the `weighted` alias to one API and tallies the resolved-model distribution from `UAIG-Resolved-Model`. The observed split should approximate the configured weights (with some random variance — this is sampling, not enforcement).

> Note: weighted random selection happens **per request** at the gateway. With a small `N` you'll see noise — bump `N` to 50–100 for a clearer signal.

In [ ]:
N = 30   # number of requests
TARGET_API = "universal"   # one of: 'universal' | 'openai' | 'unified'

if not alias_weighted:
    utils.print_warning("No weighted alias defined — skipping.")
else:
    alias_name = alias_weighted["name"]
    weights    = alias_weighted.get("weights")
    expected_pct = {
        m: round(100 * w / sum(weights), 1)
        for m, w in zip(alias_weighted["models"], weights)
    } if weights else {}

    fn = {"universal": call_universal, "openai": call_openai, "unified": call_unified}[TARGET_API]
    counter = Counter()
    failures = 0
    for i in range(N):
        r = fn(alias_name)
        if r is None:
            utils.print_error(f"Endpoint '{TARGET_API}' not available — abort.")
            break
        if r.status_code != 200:
            failures += 1
            continue
        # Prefer the UAIG header (always the resolved model); fall back to body
        resolved = r.headers.get("UAIG-Resolved-Model") or r.json().get("model", "?")
        counter[resolved] += 1
        # tiny pause so we don't pile up against rate limits
        time.sleep(0.05)

    successful = sum(counter.values())
    print(f"\n📊 Distribution after {successful} successful calls (failures: {failures}) on '{TARGET_API}'")
    if successful:
        for model, count in counter.most_common():
            actual_pct = round(100 * count / successful, 1)
            expected   = expected_pct.get(model)
            label      = f"  {model.ljust(20)} {count:3d}  ({actual_pct:5.1f}%)"
            if expected is not None:
                label += f"   expected: {expected:5.1f}%"
            print(label)

<a id='11'></a>
### 1️⃣1️⃣ Negative Test — Unauthorized Model

Sends a model that the access contract does NOT include in `allowedModels`. Expected outcome: HTTP **403 Forbidden** with `unauthorized_model_access` error code from the `validate-model-access` fragment.

In [ ]:
# Pick any model from the deployed backends that is NOT in our contract's allowed list
blocked_candidates = [m for m in all_models if m not in contract["allowed_models"]]
if not blocked_candidates:
    utils.print_warning("No deployed model exists outside the contract's allowed list — skipping.")
else:
    blocked_model = blocked_candidates[0]
    print(f"Sending blocked model: '{blocked_model}'")
    r = call_universal(blocked_model)
    if r is None:
        utils.print_warning("Endpoint not available — skipped.")
    else:
        print(f"  status: {r.status_code}")
        if r.status_code == 403:
            utils.print_ok("  ✓ 403 Forbidden — RBAC working as expected.")
            try:
                err = r.json().get("error", {})
                print(f"  code: {err.get('code')}")
                print(f"  message: {err.get('message')}")
            except Exception:
                pass
        else:
            utils.print_warning(f"  Unexpected status: body={r.text[:200]}")

<a id='summary'></a>
### 📋 Summary

If everything ran green, you've validated:

- ✅ `resolve-model-alias` fragment is deployed and inlines the configured aliases at deploy time.
- ✅ All 3 APIs (Azure OpenAI, Universal LLM, Unified AI) honor the same alias map consistently.
- ✅ `priority` strategy resolves to the first underlying model deterministically (and supports cross-model fallback in the Unified AI retry block).
- ✅ `weighted` strategy distributes requests across underlying models proportional to weights.
- ✅ Direct model selection still works unchanged — the resolver is a no-op on non-alias models.
- ✅ RBAC at the access-contract level is enforced against the **alias name**, not the underlying real models.
- ✅ `GET /deployments` (model discovery) returns aliases as first-class entries with a `properties.capabilities.description` describing the underlying real models and routing strategy, and is filtered by the contract's `allowedModels` so unauthorized real models are hidden.
- ✅ `UAIG-*` debug headers (`UAIG-Alias`, `UAIG-Resolved-Model`, `UAIG-Backend`, ...) provide end-to-end visibility into the gateway's routing decision.

### Next Steps

- Tweak the `weights` and re-run the distribution test to confirm sampling tracks the new weights.
- Edit `model_aliases` to add a real production alias (e.g. `embeddings`, `reasoning`) and re-deploy step 3.
- Use the `UAIG-Request-Id` header to correlate against APIM trace logs in App Insights when you need to diagnose a routing decision.


<a id='cleanup'></a>
### 🧹 Cleanup (Optional)

Removes only the access contract created above (the LLM backends and the `resolve-model-alias` fragment are intended to persist). Run this if you want to delete the test product/subscription.

In [ ]:
do_cleanup = False   # flip to True to actually delete

if not do_cleanup:
    utils.print_info("Cleanup skipped (set do_cleanup=True to remove the access contract).")
else:
    product_id = f"LLM-{contract['business_unit']}-{contract['use_case_name']}-{contract['environment']}"
    utils.run(
        f"az apim product delete -g {governance_hub_resource_group} -n {apim_resource_name} "
        f"--product-id {product_id} --delete-subscriptions true --yes",
        f"Deleted product {product_id}",
        f"Could not delete product {product_id}",
    )